# Week 2: Deep Learning — CNN for Cancer Detection
**FURP 2026 — Cancer Detection Using Artificial Intelligence**  
**Instructor: Prof. Elio Espejo**

---

## 本周目标
1. 搭建卷积神经网络（CNN）自动学习特征
2. 数据增强（Data Augmentation）提升泛化能力
3. 训练循环：前向传播 / 反向传播 / 梯度下降
4. AI Assistant Tasks：迁移学习 / 学习率调度 / 特征图可视化 / 模型对比

## Week 1 vs Week 2 核心区别
| | Week 1 随机森林 | Week 2 CNN |
|--|----------------|------------|
| 特征 | 人工设计 19 个 | 网络自动学习 |
| 输入 | (500, 19) 特征矩阵 | (N, 1, 224, 224) 像素矩阵 |
| 核心算法 | 信息熵 + Bootstrap | 梯度下降 + 反向传播 |

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{device}')
plt.rcParams['figure.dpi'] = 120

---
## Step 1: 数据准备（复用 Week 1 的合成数据）

In [ ]:
def create_synthetic_mammograms(num_samples=500):
    np.random.seed(42)
    images, labels = [], []
    image_size = 256
    y, x = np.ogrid[:image_size, :image_size]
    for i in range(num_samples):
        image = np.random.normal(120, 25, (image_size, image_size))
        image += 20 * np.sin(x/30) * np.cos(y/25)
        has_cancer = np.random.random() < 0.3
        if has_cancer:
            cx = np.random.randint(40, image_size-40)
            cy = np.random.randint(40, image_size-40)
            r  = np.random.randint(15, 25)
            for dy in range(-r, r):
                for dx in range(-r, r):
                    if dx**2+dy**2 < r**2 and 0<=cy+dy<image_size and 0<=cx+dx<image_size:
                        image[cy+dy, cx+dx] += 60 + np.random.normal(0, 5)
            labels.append(1)
        else:
            labels.append(0)
        image = np.clip(image + np.random.normal(0,10,image.shape), 0, 255).astype(np.uint8)
        images.append(image)
    images = np.array(images)
    labels = np.array(labels)
    print(f'数据集：{len(images)}张 | 癌症：{np.sum(labels)} | 正常：{np.sum(labels==0)}')
    return images, labels

images, labels = create_synthetic_mammograms(500)

---
## Step 2: Dataset + 数据增强

**数据增强原理：** 人为创造训练数据的变体，模拟真实临床变化：
- 随机旋转：模拟不同拍摄角度
- 水平翻转：利用乳腺左右对称性
- 颜色抖动：模拟不同成像设备的亮度差异

In [ ]:
class MedicalImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images.astype(np.uint8)
        self.labels    = labels.astype(np.int64)
        self.transform = transform

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx], mode='L')
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0
        return image, torch.tensor(self.labels[idx], dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

X_tmp, X_test, y_tmp, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

print(f'训练:{len(X_train)} | 验证:{len(X_val)} | 测试:{len(X_test)}')

train_loader = DataLoader(MedicalImageDataset(X_train, y_train, train_transform), batch_size=16, shuffle=True)
val_loader   = DataLoader(MedicalImageDataset(X_val,   y_val,   val_transform),   batch_size=16, shuffle=False)
test_loader  = DataLoader(MedicalImageDataset(X_test,  y_test,  val_transform),   batch_size=16, shuffle=False)

---
## Step 3: CNN 架构

**卷积层数学：**
$$\text{output}[i,j] = \sum_m \sum_n \text{input}[i+m, j+n] \times \text{kernel}[m,n]$$

**BatchNorm：** $\hat{x} = \frac{x-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}$

**ReLU：** $f(x) = \max(0, x)$

**特征图尺寸变化：** 224 → 112 → 56 → 28 → 1×1（全局平均池化）

In [ ]:
class MedicalCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(MedicalCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1: 基础边缘检测 (1→32通道, 224→112)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 2: 纹理识别 (32→64通道, 112→56)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 3: 复杂模式 (64→128通道, 56→28)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 4: 高层特征 (128→256通道, 28→1×1)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256, 128),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = MedicalCNN(num_classes=2).to(device)
print(f'模型参数量：{model.count_parameters():,}')

# 测试前向传播
test_input = torch.randn(4, 1, 224, 224).to(device)
test_out   = model(test_input)
print(f'输入形状：{test_input.shape}  →  输出形状：{test_out.shape}')

---
## Step 4: 训练循环

**交叉熵损失：** $L = -\sum_c y_c \log(\hat{p}_c)$

**Adam 优化器：** 自适应学习率，结合 Momentum + RMSProp

**早停机制：** 验证准确率连续 10 轮不提升则停止

In [ ]:
# 加权损失函数（处理类别不平衡）
num_normal = np.sum(labels == 0)
num_cancer = np.sum(labels == 1)
w_normal = len(labels) / (2 * num_normal)
w_cancer = len(labels) / (2 * num_cancer)
class_weights = torch.FloatTensor([w_normal, w_cancer]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f'类别权重 → 正常:{w_normal:.2f}  癌症:{w_cancer:.2f}')

optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc   = 0.0
patience_count = 0
max_patience   = 10
num_epochs     = 30

print(f'\n{"Epoch":>5} | {"训练损失":>8} | {"训练准确":>8} | {"验证损失":>8} | {"验证准确":>8}')
print('-' * 55)

for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss   = criterion(output, target)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        t_total   += target.size(0)
        t_correct += (predicted == target).sum().item()

    # 验证阶段
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            v_loss += criterion(output, target).item()
            _, predicted = torch.max(output.data, 1)
            v_total   += target.size(0)
            v_correct += (predicted == target).sum().item()

    avg_t_loss = t_loss / len(train_loader)
    avg_v_loss = v_loss / len(val_loader)
    t_acc = 100 * t_correct / t_total
    v_acc = 100 * v_correct / v_total

    history['train_loss'].append(avg_t_loss)
    history['val_loss'].append(avg_v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    scheduler.step(avg_v_loss)
    print(f'{epoch+1:5d} | {avg_t_loss:8.4f} | {t_acc:7.1f}% | {avg_v_loss:8.4f} | {v_acc:7.1f}%')

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_count = 0
        torch.save(model.state_dict(), 'best_medical_cnn.pth')
    else:
        patience_count += 1
        if patience_count >= max_patience:
            print(f'\n早停触发：第 {epoch+1} 轮')
            break

print(f'\n最佳验证准确率：{best_val_acc:.1f}%')

---
## AI Assistant Task 1: 迁移学习（Transfer Learning）

**原理：** 用 ImageNet 预训练权重初始化网络，利用已学到的通用视觉特征（边缘/纹理/形状）迁移到医学图像。

$$\text{Fine-tuning}: \theta_{\text{medical}} = \theta_{\text{ImageNet}} - \alpha \nabla L_{\text{medical}}$$

In [ ]:
import torchvision.models as models

class TransferLearningCNN(nn.Module):
    """
    基于 ResNet18 的迁移学习模型
    策略：冻结前几层（通用特征），只训练后几层（医学专有特征）
    """
    def __init__(self, num_classes=2, freeze_layers=6):
        super(TransferLearningCNN, self).__init__()
        # 加载预训练 ResNet18
        resnet = models.resnet18(weights=None)  # 不联网，用随机权重演示
        # 修改第一层：ImageNet是RGB(3通道)，医学图像是灰度(1通道)
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # 冻结前N层（固定通用特征）
        layers = list(resnet.children())
        for layer in layers[:freeze_layers]:
            for param in layer.parameters():
                param.requires_grad = False
        # 替换分类头
        in_features = resnet.fc.in_features
        resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.model = resnet

    def forward(self, x):
        return self.model(x)

    def count_trainable(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_total(self):
        return sum(p.numel() for p in self.parameters())


tl_model = TransferLearningCNN(num_classes=2, freeze_layers=6).to(device)
total     = tl_model.count_total()
trainable = tl_model.count_trainable()
frozen    = total - trainable

print('迁移学习模型统计：')
print(f'  总参数量：   {total:,}')
print(f'  可训练参数： {trainable:,} ({trainable/total*100:.1f}%)')
print(f'  冻结参数：   {frozen:,} ({frozen/total*100:.1f}%)')
print('\n迁移学习优势：')
print('  1. 从通用视觉特征出发，收敛更快')
print('  2. 只需少量医学数据即可获得好效果')
print('  3. 冻结层保留边缘/纹理等底层特征')

# 对比参数量
fig, ax = plt.subplots(figsize=(8, 4))
models_info = [
    ('从零训练 CNN', model.count_parameters(), 0),
    ('迁移学习（可训练）', trainable, 0),
    ('迁移学习（冻结）', frozen, 1)
]
names   = ['从零训练 CNN', '迁移学习\n可训练部分', '迁移学习\n冻结部分']
values  = [m[1] for m in models_info]
colors  = ['#185FA5', '#1D9E75', '#B4B2A9']
ax.barh(names, values, color=colors, alpha=0.85)
ax.set_xlabel('参数量', fontsize=12)
ax.set_title('迁移学习 vs 从零训练：参数量对比', fontweight='bold', fontsize=13)
for i, v in enumerate(values):
    ax.text(v + 1000, i, f'{v:,}', va='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('task1_transfer_learning.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 1 完成！')

---
## AI Assistant Task 2: 训练曲线可视化 + 学习率调度分析

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Training Curves', fontsize=16, fontweight='bold')

# 损失曲线
ax1 = axes[0]
ax1.plot(epochs_range, history['train_loss'], 'o-',
         color='#185FA5', lw=2, markersize=4, label='Training Loss')
ax1.plot(epochs_range, history['val_loss'], 's-',
         color='#E24B4A', lw=2, markersize=4, label='Validation Loss')
ax1.fill_between(epochs_range, history['train_loss'],
                 alpha=0.1, color='#185FA5')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('Loss Curves', fontweight='bold', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 准确率曲线
ax2 = axes[1]
ax2.plot(epochs_range, history['train_acc'], 'o-',
         color='#185FA5', lw=2, markersize=4, label='Training Accuracy')
ax2.plot(epochs_range, history['val_acc'], 's-',
         color='#E24B4A', lw=2, markersize=4, label='Validation Accuracy')
ax2.fill_between(epochs_range, history['val_acc'],
                 alpha=0.1, color='#E24B4A')
best_epoch = np.argmax(history['val_acc']) + 1
ax2.axvline(x=best_epoch, color='#1D9E75', linestyle='--',
            lw=2, label=f'Best epoch={best_epoch}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy Curves', fontweight='bold', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# 过拟合诊断
gap = np.array(history['train_acc']) - np.array(history['val_acc'])
max_gap = np.max(gap)
print(f'\n训练曲线分析：')
print(f'  最佳验证准确率：{max(history["val_acc"]):.1f}% (第{best_epoch}轮)')
print(f'  最大训练-验证差距：{max_gap:.1f}%')
if max_gap > 15:
    print('  ⚠️  存在过拟合，建议增加Dropout或减小模型')
elif max_gap > 5:
    print('  轻微过拟合，当前状况可接受')
else:
    print('  训练稳定，无明显过拟合')

plt.tight_layout()
plt.savefig('task2_training_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 2 完成！')

---
## AI Assistant Task 3: 特征图可视化（CNN 学到了什么）

特征图展示每个卷积核在图像上的响应——越亮的地方说明该滤波器对那个区域的激活越强。

In [ ]:
# 加载最佳模型
try:
    model.load_state_dict(torch.load('best_medical_cnn.pth', map_location=device))
except:
    pass  # 如果文件不存在，使用当前模型
model.eval()

# 选一张癌症图像
cancer_indices = np.where(y_test == 1)[0]
sample_img = X_test[cancer_indices[0]]

# 转为张量
img_tensor = val_transform(Image.fromarray(sample_img, mode='L')).unsqueeze(0).to(device)

# 提取各层特征图
feature_maps = {}
hooks = []

def make_hook(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

hooks.append(model.features[0].register_forward_hook(make_hook('Block1_Conv')))
hooks.append(model.features[4].register_forward_hook(make_hook('Block2_Conv')))
hooks.append(model.features[8].register_forward_hook(make_hook('Block3_Conv')))

with torch.no_grad():
    _ = model(img_tensor)

for h in hooks:
    h.remove()

fig, axes = plt.subplots(4, 8, figsize=(18, 10))
fig.suptitle('CNN Feature Maps: What Each Layer Sees', fontsize=16, fontweight='bold')

# 原图
for ax in axes[0]:
    ax.axis('off')
axes[0][3].imshow(sample_img, cmap='gray')
axes[0][3].set_title('Original\nCancer Image', fontsize=9, color='#E24B4A')
axes[0][3].axis('on')
axes[0][4].imshow(sample_img, cmap='gray')
axes[0][4].axis('on')

# Block1、2、3 的前8个特征图
for row_idx, (name, fmap) in enumerate([
    ('Block1 (边缘)', feature_maps.get('Block1_Conv')),
    ('Block2 (纹理)', feature_maps.get('Block2_Conv')),
    ('Block3 (模式)', feature_maps.get('Block3_Conv'))
]):
    row = row_idx + 1
    if fmap is None:
        continue
    for col in range(8):
        if col < fmap.shape[1]:
            fm = fmap[0, col].numpy()
            axes[row][col].imshow(fm, cmap='viridis')
            axes[row][col].axis('off')
            if col == 0:
                axes[row][col].set_ylabel(name, fontsize=9, rotation=90, labelpad=30)
        else:
            axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('task3_feature_maps.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 3 完成！特征图从边缘 → 纹理 → 复杂模式，层层递进')

---
## AI Assistant Task 4: Week1 vs Week2 全面对比

In [ ]:
# 评估 CNN
try:
    model.load_state_dict(torch.load('best_medical_cnn.pth', map_location=device))
except:
    pass
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        probs  = F.softmax(output, dim=1)
        _, predicted = torch.max(output, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(target.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

cnn_acc  = accuracy_score(all_labels, all_preds)
cnn_sens = recall_score(all_labels, all_preds, zero_division=0)
tn = np.sum((all_preds==0)&(all_labels==0))
fp = np.sum((all_preds==1)&(all_labels==0))
cnn_spec = tn/(tn+fp) if (tn+fp)>0 else 0
cnn_f1   = f1_score(all_labels, all_preds, zero_division=0)
cnn_auc  = roc_auc_score(all_labels, all_probs)

# Week1 基线（随机森林）
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import cv2
from skimage.feature import local_binary_pattern

def extract_features(image):
    f = []
    f.extend([np.mean(image), np.std(image), np.max(image)-np.min(image)])
    norm = (image - np.mean(image)) / (np.std(image) + 1e-7)
    f.extend([np.mean(norm**3), np.mean(norm**4), np.median(image),
               np.percentile(image,75)-np.percentile(image,25)])
    sx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    em = np.sqrt(sx**2 + sy**2)
    f.extend([np.mean(em>50), np.mean(em), np.var(em)])
    lbp = local_binary_pattern(image, 8, 1, method='uniform')
    h, _ = np.histogram(lbp, bins=10, range=(0,9))
    h = h.astype(float)/(h.sum()+1e-7)
    f.extend([np.sum(h**2), -np.sum(h*np.log2(h+1e-7))])
    _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        lc = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(lc); per = cv2.arcLength(lc, True)
        circ = 4*np.pi*area/(per**2) if per>0 else 0
        xc,yc,w,h2 = cv2.boundingRect(lc)
        hull = cv2.convexHull(lc); ha = cv2.contourArea(hull)
        f.extend([circ, w/h2 if h2>0 else 0, area/ha if ha>0 else 0])
    else:
        f.extend([0,0,0])
    fft = np.fft.fftshift(np.fft.fft2(image))
    mag = np.abs(fft)
    cy2, cx2 = mag.shape[0]//2, mag.shape[1]//2
    yc2, xc2 = np.ogrid[:mag.shape[0],:mag.shape[1]]
    lr = min(cx2,cy2)//4; hr = min(cx2,cy2)//2
    f.extend([np.mean(mag), np.std(mag),
               np.sum(mag[(xc2-cx2)**2+(yc2-cy2)**2<=lr**2]),
               np.sum(mag[(xc2-cx2)**2+(yc2-cy2)**2>=hr**2])])
    return f

print('提取Week1特征...')
feats_test = np.array([extract_features(img) for img in X_test])
feats_train = np.array([extract_features(img) for img in X_train])
sc = StandardScaler()
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(sc.fit_transform(feats_train), y_train)
rf_pred  = rf.predict(sc.transform(feats_test))
rf_proba = rf.predict_proba(sc.transform(feats_test))[:,1]

rf_sens = recall_score(y_test, rf_pred, zero_division=0)
tn2 = np.sum((rf_pred==0)&(y_test==0)); fp2 = np.sum((rf_pred==1)&(y_test==0))
rf_spec = tn2/(tn2+fp2) if (tn2+fp2)>0 else 0

print(f'\n{"="*55}')
print(f'{"指标":<15} {"Week1 RF":>15} {"Week2 CNN":>15}')
print(f'{"="*55}')
print(f'{"准确率":<15} {accuracy_score(y_test,rf_pred)*100:>14.1f}% {cnn_acc*100:>14.1f}%')
print(f'{"敏感性":<15} {rf_sens*100:>14.1f}% {cnn_sens*100:>14.1f}%')
print(f'{"特异性":<15} {rf_spec*100:>14.1f}% {cnn_spec*100:>14.1f}%')
print(f'{"AUC-ROC":<15} {roc_auc_score(y_test,rf_proba):>15.3f} {cnn_auc:>15.3f}')
print(f'{"="*55}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Week 1 (RF) vs Week 2 (CNN): Full Comparison', fontsize=15, fontweight='bold')

# 指标对比
metrics  = ['Accuracy', 'Sensitivity', 'Specificity', 'AUC-ROC']
rf_vals  = [accuracy_score(y_test,rf_pred), rf_sens, rf_spec, roc_auc_score(y_test,rf_proba)]
cnn_vals = [cnn_acc, cnn_sens, cnn_spec, cnn_auc]
x = np.arange(len(metrics)); w = 0.35
axes[0].bar(x-w/2, rf_vals,  w, label='Week1 RF',  color='#185FA5', alpha=0.85)
axes[0].bar(x+w/2, cnn_vals, w, label='Week2 CNN', color='#E24B4A', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics, rotation=15, fontsize=9)
axes[0].set_ylabel('Score', fontsize=11); axes[0].set_ylim([0, 1.15])
axes[0].set_title('Metric Comparison', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3, axis='y')

# ROC 对比
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, rf_proba)
fpr_cnn, tpr_cnn, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr_rf,  tpr_rf,  color='#185FA5', lw=2, label=f'RF  AUC={roc_auc_score(y_test,rf_proba):.3f}')
axes[1].plot(fpr_cnn, tpr_cnn, color='#E24B4A', lw=2, label=f'CNN AUC={cnn_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1.5,alpha=0.5)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve Comparison', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

# 混淆矩阵对比
cm_cnn = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Normal','Cancer'], yticklabels=['Normal','Cancer'],
            ax=axes[2], cbar=False, annot_kws={'size':14,'weight':'bold'})
axes[2].set_title('CNN Confusion Matrix', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('task4_week1_vs_week2.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 4 完成！')

---
## Week 2 总结

| 组件 | 技术 | 作用 |
|------|------|------|
| 数据增强 | 旋转/翻转/颜色抖动 | 提升泛化能力 |
| CNN架构 | Conv+BN+ReLU+Pool | 自动学习特征 |
| 加权损失 | CrossEntropy(weight) | 处理类别不平衡 |
| 优化器 | Adam + ReduceLROnPlateau | 自适应学习率 |
| 早停 | patience=10 | 防止过拟合 |

**核心洞见：** CNN 省去了手工特征设计，直接从像素学习，理论性能上限更高。  
**下周预告：** 迁移学习 + 不确定性量化 + 集成方法！